#### Cella import dei percorsi


In [ ]:
import sys
sys.path.append("..")
import numpy as np
from pathlib import Path
from helpers import load_csv_data

data_path = Path('../data')
cache_path = data_path / 'cache'
cache_path.mkdir(exist_ok=True)   # crea la cartella se non esiste

### Caricamento con cache

In [ ]:
if (cache_path / 'x_train.npy').exists():
    # Già salvati: li leggo veloce
    x_train = np.load(cache_path / 'x_train.npy')
    x_test = np.load(cache_path / 'x_test.npy')
    y_train = np.load(cache_path / 'y_train.npy')
    train_ids = np.load(cache_path / 'train_ids.npy')
    test_ids = np.load(cache_path / 'test_ids.npy')
    print("Caricati dalla cache")
else:
    # Prima volta: leggo i CSV (lento, 1-2 minuti) e salvo
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(str(data_path))
    np.save(cache_path / 'x_train.npy', x_train)
    np.save(cache_path / 'x_test.npy', x_test)
    np.save(cache_path / 'y_train.npy', y_train)
    np.save(cache_path / 'train_ids.npy', train_ids)
    np.save(cache_path / 'test_ids.npy', test_ids)
    print("Letti dai CSV e salvati in cache")

### Header

In [ ]:
with open(data_path / 'x_train.csv', mode='r', encoding='utf-8') as f:
    headers = f.readline().strip().split(',')
headers = headers[1:]   # tolgo 'Id'

In [ ]:
print(x_train.shape, x_test.shape, y_train.shape)
print(len(headers))

## Iniziamo a togliere alcuni valori dalla tabella

Dato che 322 valori sono troppi e molte sono ingormazioni inutili, possiamo scartare colonne come:
Id, _STATE, FMONTH, IDATE, IMONTH, IDAY, IYEAR, STATERES

Amministrative (come è stata gestita la chiamata)
DISPCODE, SEQNO, _PSU, CTELENUM, PVTRESD1, CELLFON3, NUMHHOL2, CPDEMO1, QSTVER, QSTLANG, MSCODE

Pesi statistici del sondaggio (servono ai ricercatori per ricalibrare il campione, non descrivono la persona)
_STSTR, _STRWT, _RAWRAKE, _WT2RAKE, _DUALUSE, _DUALCOR, _LLCPWT

In [ ]:
da_togliere = []

da_togliere = [
    # Identificativi e date
    'Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'STATERES',
    
    # Amministrative (gestione della chiamata)
    'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'CELLFON3', 
    'NUMHHOL2', 'CPDEMO1', 'QSTVER', 'QSTLANG', 'MSCODE',
    
    # Pesi statistici del sondaggio
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_DUALUSE', '_DUALCOR', '_LLCPWT'
]

# lista degli header da tenere
# scorre la lista headers, e ottiene sia l'inidice i che il nome, tiene solo gli indici in cui il nome non è nella lista da togliere 
keep = [i for i, name in enumerate(headers) if name not in da_togliere]

# selezioniamo le colonne, ovvero prendiamo tutte le righe " : ", ma solo con le colonne i cui indici sono specificati nella lista keep creata al primo passaggio.
x_train_clean = x_train[:, keep]

#Crea una nuova lista di nomi (names) che corrisponde esattamente alle colonne rimaste in x.
names = [headers[i] for i in keep]

# Applichiamo a x_test: 
x_test_clean = x_test[:, keep]

print(x_train_clean.shape, x_test_clean.shape, len(names))

In [ ]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent            # notebooks/ -> ml-project1/
sys.path.insert(0, str(ROOT))       # per importare helpers.py e cleaning.py

from helpers import load_csv_data
from cleaning import (read_header, fill_structural_zeros, recode_none_to_zero,
                      find_special_codes, special_codes_to_nan)

data_path = ROOT / "Data"           # cartella con x_train.csv, y_train.csv, x_test.csv
cache_path = ROOT / "cache"         # cartella per i file .npy
cache_path.mkdir(exist_ok=True)

if (cache_path / 'x_train.npy').exists():
    # Già salvati: li leggo veloce
    x_train = np.load(cache_path / 'x_train.npy')
    x_test = np.load(cache_path / 'x_test.npy')
    y_train = np.load(cache_path / 'y_train.npy')
    train_ids = np.load(cache_path / 'train_ids.npy')
    test_ids = np.load(cache_path / 'test_ids.npy')
    print("Caricati dalla cache")
else:
    # Prima volta: leggo i CSV (lento, 1-2 minuti) e salvo
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(str(data_path))
    np.save(cache_path / 'x_train.npy', x_train)
    np.save(cache_path / 'x_test.npy', x_test)
    np.save(cache_path / 'y_train.npy', y_train)
    np.save(cache_path / 'train_ids.npy', train_ids)
    np.save(cache_path / 'test_ids.npy', test_ids)
    print("Letti dai CSV e salvati in cache")

# nomi delle colonne: load_csv_data toglie già la colonna 'Id'
cols = read_header(data_path / "x_train.csv")
if len(cols) == x_train.shape[1] + 1:
    cols = cols[1:]
assert len(cols) == x_train.shape[1]

print("train:", x_train.shape, " test:", x_test.shape)
print(f"NaN iniziali: {np.isnan(x_train).mean():.1%}")